In [11]:
import pandas as pd
import os

ret = pd.read_csv("../data/retail.csv")
save_dir = "../data/sampled"
os.makedirs(save_dir, exist_ok=True)

random_state = 42
unique_fkeys = pd.Series(ret['fkey'].unique())
test_fkeys = unique_fkeys.sample(n=3000, random_state=random_state)
test_df = ret[ret['fkey'].isin(test_fkeys)]

test_csv = os.path.join(save_dir, "retail_test_3000.csv")
test_df.to_csv(test_csv, index=False)
print(f"테스트용 fkey 3000개 샘플링 완료: {test_csv}")

sample_sizes = [10, 100, 500, 1000, 3000, 5000, 10000, 15000]
train_fkey_candidates = unique_fkeys[~unique_fkeys.isin(test_fkeys)] # 학습용 fkey 후보 (test 제외)

for size in sample_sizes:
    sampled_fkeys = train_fkey_candidates.sample(n=size, random_state=random_state)
    sampled_df = ret[ret['fkey'].isin(sampled_fkeys)]
    out_csv = os.path.join(save_dir, f"retail_sample_{size}.csv")
    sampled_df.to_csv(out_csv, index=False)
    print(f"학습용 fkey 그룹 {size}개 샘플링 완료: {out_csv}")

테스트용 fkey 3000개 샘플링 완료: ../data/sampled/retail_test_3000.csv
학습용 fkey 그룹 10개 샘플링 완료: ../data/sampled/retail_sample_10.csv
학습용 fkey 그룹 100개 샘플링 완료: ../data/sampled/retail_sample_100.csv
학습용 fkey 그룹 500개 샘플링 완료: ../data/sampled/retail_sample_500.csv
학습용 fkey 그룹 1000개 샘플링 완료: ../data/sampled/retail_sample_1000.csv
학습용 fkey 그룹 3000개 샘플링 완료: ../data/sampled/retail_sample_3000.csv
학습용 fkey 그룹 5000개 샘플링 완료: ../data/sampled/retail_sample_5000.csv
학습용 fkey 그룹 10000개 샘플링 완료: ../data/sampled/retail_sample_10000.csv
학습용 fkey 그룹 15000개 샘플링 완료: ../data/sampled/retail_sample_15000.csv


In [12]:
from sdv.metadata import Metadata

data_dir = "../data/sampled"
meta_cols = ['fkey', 'date', 'CustomerID', 'Country']
flow_cols = ['fkey', 'Description', 'Quantity', 'UnitPrice']
datetime_format = '%Y-%m-%d %H:%M:%S'

def process_and_save_meta_flow(csv_path, prefix):
    df = pd.read_csv(csv_path)
    meta_df = df[meta_cols].drop_duplicates().copy()
    flow_df = df[flow_cols].copy()

    meta_path = os.path.join(save_dir, f"{prefix}_meta.csv")
    flow_path = os.path.join(save_dir, f"{prefix}_flow.csv")
    meta_df.to_csv(meta_path, index=False)
    flow_df.to_csv(flow_path, index=False)

    metadata = Metadata.detect_from_dataframes({"meta": meta_df, "flow": flow_df})
    metadata.update_column(column_name='fkey', table_name='meta', sdtype='id')
    metadata.update_column(column_name='fkey', table_name='flow', sdtype='id')
    metadata.update_column(column_name='date', table_name='meta', sdtype='datetime', datetime_format=datetime_format)
    meta_json_path = os.path.join(save_dir, f"{prefix}_metadata.json")
    metadata.save_to_json(meta_json_path)
    print(f"{prefix} 저장 완료 (meta, flow, metadata)")

for size in sample_sizes:
    sample_csv = os.path.join(save_dir, f"retail_sample_{size}.csv")
    process_and_save_meta_flow(sample_csv, f"retail_sample_{size}")

test_csv = os.path.join(save_dir, "retail_test_3000.csv")
process_and_save_meta_flow(test_csv, "retail_test_3000")

retail_sample_10: 저장 완료 (meta, flow, metadata)
retail_sample_100: 저장 완료 (meta, flow, metadata)
retail_sample_500: 저장 완료 (meta, flow, metadata)
retail_sample_1000: 저장 완료 (meta, flow, metadata)
retail_sample_3000: 저장 완료 (meta, flow, metadata)
retail_sample_5000: 저장 완료 (meta, flow, metadata)
retail_sample_10000: 저장 완료 (meta, flow, metadata)
retail_sample_15000: 저장 완료 (meta, flow, metadata)
retail_test_3000: 저장 완료 (meta, flow, metadata)
